In [1]:

from terratorch.registry import BACKBONE_REGISTRY
import rasterio
import numpy as np
import torch
from torchvision import transforms


INFO:albumentations.check_version:A new version of Albumentations is available: 2.0.8 (you have 1.4.10). Upgrade using: pip install --upgrade albumentations


In [2]:
# Model settings
device = "cuda" if torch.cuda.is_available() else "cpu"

model = BACKBONE_REGISTRY.build(
    "terramind_v1_base", # terramind_v1_base
    pretrained=True,
    modalities=["RGB"]
).to(device)

model.eval()

INFO:httpx:HTTP Request: HEAD https://huggingface.co/ibm-esa-geospatial/TerraMind-1.0-base/resolve/main/TerraMind_v1_base.pt "HTTP/1.1 302 Found"
INFO:httpx:HTTP Request: GET https://huggingface.co/api/models/ibm-esa-geospatial/TerraMind-1.0-base/xet-read-token/fb96c70d0a5f68dcc44030b89cbfd8ec3fb0c67a "HTTP/1.1 200 OK"


TerraMind_v1_base.pt:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

C:\Users\Isabelle\anaconda3\envs\terramind\Lib\site-packages\huggingface_hub\file_download.py:130: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Isabelle\.cache\huggingface\hub\models--ibm-esa-geospatial--TerraMind-1.0-base. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


TerraMindViT(
  (encoder_embeddings): ModuleDict(
    (untok_sen2rgb@224): ImageEncoderEmbedding(
      (proj): Linear(in_features=768, out_features=768, bias=False)
    )
  )
  (encoder): ModuleList(
    (0-11): 12 x Block(
      (norm1): LayerNorm()
      (attn): Attention(
        (qkv): Linear(in_features=768, out_features=2304, bias=False)
        (attn_drop): Dropout(p=0.0, inplace=False)
        (proj): Linear(in_features=768, out_features=768, bias=False)
        (proj_drop): Dropout(p=0.0, inplace=False)
      )
      (drop_path): Identity()
      (norm2): LayerNorm()
      (mlp): GatedMlp(
        (fc1): Linear(in_features=768, out_features=2048, bias=False)
        (act): SiLU()
        (fc2): Linear(in_features=2048, out_features=768, bias=False)
        (fc3): Linear(in_features=768, out_features=2048, bias=False)
      )
    )
  )
  (encoder_norm): LayerNorm()
  (tokenizer): ModuleDict()
)

In [5]:
image_fp = '../../../images/69_1.tiff'

In [11]:
import torchvision

dict_normalize = {
    'mean': [0.11866877228021622, 0.09809420257806778, 0.06679389625787735],
    'std': [0.030054444447159767, 0.02086435630917549, 0.018123868852853775]
}

transform = torchvision.transforms.Compose([
        torchvision.transforms.CenterCrop(224),
        torchvision.transforms.Normalize(mean=dict_normalize['mean'], std=dict_normalize['std']),
    ])

In [12]:
tile = []

with rasterio.open(image_fp) as src:

    bands = src.descriptions
    bands_indexes = []

    for b in ['red', 'green', 'blue']:
        bands_indexes.append(bands.index(b) + 1)

    for band in bands_indexes:
        layer = src.read(band)
        tile.append(layer)

    tile_t = np.stack(tile, axis=0)
    tile_t = np.nan_to_num(tile_t)

image = transform(torch.tensor(tile_t, dtype=torch.float32))
image = image[None, :,:,:]
image = image.float()

In [21]:
image_dict = {"RGB": image}
image.shape

torch.Size([1, 3, 224, 224])

In [15]:
s1_out = model(image_dict)
len(s1_out)

12

In [23]:
s1_out[-1].shape

torch.Size([1, 196, 768])